<a href="https://colab.research.google.com/github/Estherbh/Classi-fy/blob/main/organize_farmers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Farmer list organisation: names and codes

Cleans a raw list of farmers (name + identification code) typed by hand,
and produces a usable Excel file.

## ⚠️ Personal data

**The real data is not in this notebook.** It is read from
`data/fermiers_brut.txt`, which is excluded from git via `.gitignore`.

This file contains the names of real people and their codes: it must never
be committed or published. On Colab, upload it manually into the session.

## Expected format

One entry per line. Input can be irregular, the script tolerates it
(the codes below are fictitious):

```
1.SURNAME FIRSTNAME:PH-MGN-XX-0000
2 SURNAME FIRSTNAME PH-MTG-XX 000
3. SURNAME FIRSTNAME              <- name without a code, accepted
```


In [ ]:
from pathlib import Path
import re

import pandas as pd

# Source file: outside git, contains personal data.
FICHIER_SOURCE = Path("data/fermiers_brut.txt")
FICHIER_SORTIE = Path("data/liste_fermiers_avec_codes.xlsx")

if not FICHIER_SOURCE.exists():
    raise FileNotFoundError(
        f"{FICHIER_SOURCE} not found.\n"
        "Put the raw farmer list in this file (one entry per line).\n"
        "On Colab: upload it into the session, it must not be in the repository."
    )

lignes = [
    ligne.strip()
    for ligne in FICHIER_SOURCE.read_text(encoding="utf-8").splitlines()
    if ligne.strip()
]

print(f"{len(lignes)} lines read from {FICHIER_SOURCE}")

In [ ]:
# A code follows the pattern <PH|CO>-<MGN|MTG|MTH>-<zone>-<number>, for example
# PH-MGN-XX-0000 or CO-MTG-XX 000 (fictitious examples).
# Real input is irregular: spaces in the middle, PH prefix sometimes missing,
# missing hyphens. These variants are accepted, then normalised, keeping the raw
# code so that what normalisation changed can be checked.
MOTIF_CODE = re.compile(r"((?:PH|CO)?\s*-?\s*(?:MGN|MTG|MTH)[\s\w-]*)$", re.IGNORECASE)


def nettoyer_nom(nom):
    """Removes leading numbering, trailing punctuation and repeated spaces."""
    nom = re.sub(r"^\s*\d+\s*[.:)\-]*\s*", "", nom)
    nom = re.sub(r"[\s.:,\-]+$", "", nom)
    nom = re.sub(r"\s+", " ", nom).strip()
    return nom.title()


def normaliser_code(code):
    """Removes internal spaces: "PH-MTG- MU- N026" -> "PH-MTG-MU-N026"."""
    return re.sub(r"\s+", "", code).strip(" .-:").upper()


def analyser_ligne(ligne):
    """Returns (name, raw_code). raw_code is None if the line has no code."""
    correspondance = MOTIF_CODE.search(ligne)
    if not correspondance:
        return nettoyer_nom(ligne), None
    code_brut = correspondance.group(1).strip()
    nom = nettoyer_nom(ligne[: correspondance.start()])
    return nom, code_brut

In [ ]:
# Column names stay in French: the Excel file is used by the field team.
enregistrements = []
for ligne in lignes:
    nom, code_brut = analyser_ligne(ligne)
    if not nom and not code_brut:
        continue
    enregistrements.append(
        {
            "Nom du Fermier": nom,
            "Code": normaliser_code(code_brut) if code_brut else "",
            "Code brut": code_brut or "",
            "Ligne source": ligne,
        }
    )

df = pd.DataFrame(enregistrements)

print(f"Total entries: {len(df)}")
print(f"  with code: {(df['Code'] != '').sum()}")
print(f"  without code: {(df['Code'] == '').sum()}")
df.head(10)

In [ ]:
# Quality checks.
# The previous version of this script did not run them: duplicates slipped
# silently into the final Excel file.

codes = df.loc[df["Code"] != "", "Code"]
codes_dupliques = codes[codes.duplicated(keep=False)]
if len(codes_dupliques):
    print(f"⚠️  {codes_dupliques.nunique()} code(s) assigned to several farmers:")
    colonnes = ["Nom du Fermier", "Code", "Ligne source"]
    print(df[df["Code"].isin(codes_dupliques)].sort_values("Code")[colonnes].to_string(index=False))
else:
    print("✅ No duplicate code.")

noms = df.loc[df["Nom du Fermier"] != "", "Nom du Fermier"]
noms_dupliques = noms[noms.duplicated(keep=False)]
if len(noms_dupliques):
    print(f"\n⚠️  {noms_dupliques.nunique()} name(s) appearing more than once:")
    for nom in sorted(noms_dupliques.unique()):
        print(f"   - {nom}")
else:
    print("\n✅ No duplicate name.")

sans_nom = df[df["Nom du Fermier"] == ""]
if len(sans_nom):
    print(f"\n⚠️  {len(sans_nom)} line(s) without a usable name:")
    for ligne in sans_nom["Ligne source"]:
        print(f"   - {ligne}")

modifies = df[(df["Code"] != "") & (df["Code"] != df["Code brut"].str.upper())]
if len(modifies):
    print(f"\nℹ️  {len(modifies)} code(s) normalised (spaces/case), to check:")
    print(modifies[["Nom du Fermier", "Code brut", "Code"]].to_string(index=False))

In [ ]:
FICHIER_SORTIE.parent.mkdir(parents=True, exist_ok=True)
df.to_excel(FICHIER_SORTIE, index=False, engine="openpyxl")
print(f"File created: {FICHIER_SORTIE}  ({len(df)} farmers)")

# On Colab only: triggers the file download.
try:
    from google.colab import files

    files.download(str(FICHIER_SORTIE))
except ImportError:
    pass